# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * Modify `__getitem__` implemented above to support slicing.
        

3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [3]:
class matrix:
    def __init__(self, n, m=None):
        if m is None:
            self.data = [row.copy() for row in n]
            if self.data and any(len(row) != len(self.data[0]) for row in self.data):
                raise ValueError
        else:
            self.data = [[0 for _ in range(m)] for _ in range(n)]

    def __getitem__(self, key):
        if isinstance(key, int):
            return self.data[key]
        if isinstance(key, tuple) and len(key) == 2:
            rows, cols = key
            if isinstance(rows, int) and isinstance(cols, int):
                return self.data[rows][cols]
            row_part = self.data[rows] if isinstance(rows, slice) else [self.data[rows]]
            if isinstance(cols, int):
                return matrix([[row[cols]] for row in row_part])
            return matrix([row[cols] for row in row_part])
        raise TypeError

    def __setitem__(self,key,value):
        if isinstance(key,tuple) and len(key) == 2 and isinstance(key[0], int) and isinstance(key[1], int):
            self.data[key[0]][key[1]] = value
            return
            
        if isinstance(key,int):
            if not isinstance(value,list) or len(value) != self.shape()[1]:
                raise ValueError
            self.data[key] = value.copy()
            return
            
        if isinstance(key,tuple) and len(key) == 2:
            rows, cols = key
            value = value.to_list() if isinstance(value,matrix) else value
            row_ids = list(range(*rows.indices(self.shape()[0]))) if isinstance(rows,slice) else [rows]
            col_ids = list(range(*cols.indices(self.shape()[1]))) if isinstance(cols,slice) else [cols]
            if len(value) != len(row_ids) or any(len(row) != len(col_ids) for row in value):
                raise ValueError
            for i, r in enumerate(row_ids):
                for j, c in enumerate(col_ids):
                    self.data[r][c] = value[i][j]
            return
        raise TypeError

    def shape(self):
        return (len(self.data), len(self.data[0]) if self.data else 0)

    def transpose(self):
        rows, cols = self.shape()
        return matrix ([[self.data[i][j] for i in range(rows)] for j in range (cols)])

    def row(self, n):
        return matrix([self.data[n]])

    def column(self,n):
        return matrix([[row[n]] for row in self.data])

    def to_list(self):
        return [row.copy() for row in self.data]

    def block(self,n_0,n_1,m_0,m_1):
        return self[m_0:m_1, n_0:n_1]

    def scalarmul(self,c):
        return matrix([[c * value for value in row] for row in self.data])

    def add(self,N):
        if self.shape() != N.shape():
            raise ValueError
        rows, cols = self.shape()
        return matrix([[self.data[i][j] + N.data[i][j] for j in range(cols)] for i in range(rows)])

    def sub(self,N):
        if self.shape() != N.shape():
            raise ValueError
        rows, cols = self.shape()
        return matrix([[self.data[i][j] - N.data[i][j] for j in range(cols)] for i in range(rows)])

    def mat_mult(self,N):
        if self.shape()[1] != N.shape()[0]:
            raise ValueError
        rows, inner = self.shape()
        _, cols = N.shape()
        return matrix([[sum(self.data[i][k] * N.data[k][j] for k in range(inner)) for j in range(cols)] for i in range(rows)])

    def element_mult(self, N):
        if self.shape() != N.shape():
            raise ValueError
        rows, cols = self.shape()
        return matrix([[self.data[i][j] * N.data[i][j] for j in range(cols)] for i in range(rows)])

    def equals(self, N):
        return isinstance(N, matrix) and self.data == N.data

    def __mul__(self, other):
        if isinstance(other, matrix):
            return self.mat_mult(other)
        return self.scalarmul(other)

    def __rmul__(self, other):
        return self.scalarmul(other)

    def __add__(self, other):
        return self.add(other)

    def __sub__(self, other):
        return self.sub(other)

    def __eq__(self, other):
        return self.equals(other)

def constant(n,m,c):
    return matrix([[float(c) for _ in range(m)] for _ in range(n)])

def zeros(n,m):
    return constant(n,m,0)

def ones(n,m):
    return constant(n,m,1)

def eye(n):
    return matrix([[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)])

M = matrix(2, 3)
assert M.to_list() == [[0, 0, 0], [0, 0, 0]]

N = matrix([[1, 2], [3, 4]])
assert N.to_list() == [[1, 2], [3, 4]]

try:
    matrix([[1, 2], [3]])
    assert False
except ValueError:
    pass
assert N[0][1] == 2
assert N[1, 0] == 3
N[0][1] = 5
N[1, 0] = 6
assert N[0, 1] == 5
assert N[1][0] == 6
A = matrix([[1, 2, 3], [4, 5, 6]])
assert A.shape() == (2, 3)
assert A.transpose().to_list() == [[1, 4], [2, 5], [3, 6]]
assert A.row(1).to_list() == [[4, 5, 6]]
assert A.column(2).to_list() == [[3], [6]]
assert A.to_list() == [[1, 2, 3], [4, 5, 6]]
assert A.block(1, 3, 0, 2).to_list() == [[2, 3], [5, 6]]
assert A[0:2, 1:3].to_list() == [[2, 3], [5, 6]]
B = matrix([[9, 8], [7, 6]])
N[:, :] = B
assert N.equals(B)
N[:, :] = [[1, 2], [3, 4]]
assert N.to_list() == [[1, 2], [3, 4]]

try:
    N[:, :] = [[1, 2, 3]]
    assert False
except ValueError:
    pass
assert constant(2, 2, 3).to_list() == [[3.0, 3.0], [3.0, 3.0]]
assert zeros(2, 2).to_list() == [[0.0, 0.0], [0.0, 0.0]]
assert ones(2, 2).to_list() == [[1.0, 1.0], [1.0, 1.0]]
assert eye(3).to_list() == [[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]

P = matrix([[1, 2], [3, 4]])
Q = matrix([[5, 6], [7, 8]])

assert P.scalarmul(2).to_list() == [[2, 4], [6, 8]]
assert P.add(Q).to_list() == [[6, 8], [10, 12]]
assert P.sub(Q).to_list() == [[-4, -4], [-4, -4]]
assert P.mat_mult(Q).to_list() == [[19, 22], [43, 50]]
assert P.element_mult(Q).to_list() == [[5, 12], [21, 32]]
assert P.equals(matrix([[1, 2], [3, 4]]))

try:
    P.add(matrix([[1, 2, 3]]))
    assert False
except ValueError:
    pass

try:
    P.mat_mult(matrix([[1, 2, 3]]))
    assert False
except ValueError:
    pass

assert 2 * P == P * 2
assert P + Q == matrix([[6, 8], [10, 12]])
assert P - Q == matrix([[-4, -4], [-4, -4]])
assert P * Q == matrix([[19, 22], [43, 50]])
assert P == matrix([[1, 2], [3, 4]])
assert P != Q

A = matrix([[2, 1], [0, 3]])
B = matrix([[4, 2], [1, 5]])

assert (A * B) * A == A * (B * A)
assert A * (B + A) == A * B + A * A
assert A * B != B * A
assert A * eye(2) == A
print("A =", A.to_list())
print("B =", B.to_list())
print("AB =", (A * B).to_list())
print("BA =", (B * A).to_list())
print("(AB)A =", ((A * B) * A).to_list())
print("A(BA) =", (A * (B * A)).to_list())
print("A(B+A) =", (A * (B + A)).to_list())
print("AB + A*A =", ((A * B) + (A * A)).to_list())
print("AB != BA:", A * B != B * A)
print("AI = A:", A * eye(2) == A)
print("All Lab 5 matrix tests passed!")

A = [[2, 1], [0, 3]]
B = [[4, 2], [1, 5]]
AB = [[9, 9], [3, 15]]
BA = [[8, 10], [2, 16]]
(AB)A = [[18, 36], [6, 48]]
A(BA) = [[18, 36], [6, 48]]
A(B+A) = [[13, 14], [3, 24]]
AB + A*A = [[13, 14], [3, 24]]
AB != BA: True
AI = A: True
All Lab 5 matrix tests passed!
